In [7]:
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import StackingRegressor
from xgboost import XGBRegressor
import math
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from xgboost import XGBRegressor

In [8]:
import glob

### Does imputing  with missing values with spatial information from a nearby buoy (44095) improve forecasting at the target station

In [9]:
na_vals = [99, 999, 9999, 99999]

def read_ndbc_stdmet(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        header = None
        for line in f:
            if line.startswith("#YY"):
                header = line.lstrip("#").strip().split()
                break
    if header is None:
        raise ValueError(f"Couldn't find #YY header in {path}")

    df = pd.read_csv(
        path,
        sep=r"\s+",
        comment="#",
        header=None,
        names=header,
        na_values=na_vals,
        engine="python",
    )
    return df

def build_station_df(station_prefix):
    files = sorted(glob.glob(f"{station_prefix}*.txt"))  
    dfs = [read_ndbc_stdmet(f) for f in files]
    df = pd.concat(dfs, ignore_index=True)

    # make datetime
    df = df.rename(columns={"YY":"year","MM":"month","DD":"day","hh":"hour","mm":"minute"})
    df["datetime"] = pd.to_datetime(df[["year","month","day","hour","minute"]], errors="coerce")
    df = df.dropna(subset=["datetime"]).set_index("datetime").sort_index()

    return df

target = build_station_df("41025h")   # target station
nbr    = build_station_df("44095h")   # nearby station


In [11]:
target.head()

,year,month,day,hour,minute,WDIR,WSPD,GST,WVHT,DPD,APD,MWD,PRES,ATMP,WTMP,DEWP,VIS,TIDE
datetime,,,,,,,,,,,,,,,,,,
2013-12-31 23:50:00,2013,12,31,23,50,305.0,7.0,10.0,0.96,5.56,4.41,2.0,1025.5,11.8,23.3,1.8,NaN,NaN
2014-01-01 00:50:00,2014,1,1,0,50,301.0,6.8,9.5,0.96,6.25,4.40,360.0,1026.6,11.8,23.3,1.4,NaN,NaN
2014-01-01 01:50:00,2014,1,1,1,50,305.0,7.5,9.5,0.89,5.88,4.23,2.0,1027.6,11.9,23.3,0.5,NaN,NaN
2014-01-01 02:50:00,2014,1,1,2,50,308.0,6.6,9.4,0.89,5.88,4.15,3.0,1027.9,11.7,23.3,-0.1,NaN,NaN
2014-01-01 03:50:00,2014,1,1,3,50,322.0,7.0,10.1,0.92,6.25,4.38,26.0,1027.9,12.0,23.2,-0.3,NaN,NaN


In [12]:
nbr.head()

,year,month,day,hour,minute,WDIR,WSPD,GST,WVHT,DPD,APD,MWD,PRES,ATMP,WTMP,DEWP,VIS,TIDE
datetime,,,,,,,,,,,,,,,,,,
2014-01-01 00:17:00,2014,1,1,0,17,NaN,NaN,NaN,0.93,5.88,4.22,28.0,NaN,NaN,14.0,NaN,NaN,NaN
2014-01-01 00:47:00,2014,1,1,0,47,NaN,NaN,NaN,0.90,6.67,4.13,48.0,NaN,NaN,14.0,NaN,NaN,NaN
2014-01-01 01:17:00,2014,1,1,1,17,NaN,NaN,NaN,0.88,6.25,4.25,45.0,NaN,NaN,14.0,NaN,NaN,NaN
2014-01-01 01:47:00,2014,1,1,1,47,NaN,NaN,NaN,0.80,6.25,4.06,44.0,NaN,NaN,14.0,NaN,NaN,NaN
2014-01-01 02:17:00,2014,1,1,2,17,NaN,NaN,NaN,0.81,6.67,4.09,57.0,NaN,NaN,14.0,NaN,NaN,NaN


In [14]:
target.isna().sum()

year           0
month          0
day            0
hour           0
minute         0
WDIR        4505
WSPD        1086
GST         1089
WVHT       73607
DPD        73607
APD        73607
MWD        73961
PRES         212
ATMP        8042
WTMP        5272
DEWP       17540
VIS       117882
TIDE      117882
dtype: int64

In [15]:
nbr.isna().sum()

year          0
month         0
day           0
hour          0
minute        0
WDIR      92715
WSPD      92715
GST       92715
WVHT          0
DPD           0
APD           0
MWD         860
PRES      92715
ATMP      92715
WTMP         76
DEWP      92715
VIS       92715
TIDE      92715
dtype: int64

## Wave Height

In [34]:
y_col = "WVHT"   # wave height target

In [35]:
target_h = target.resample("H").mean()
nbr_h    = nbr.resample("H").mean()

print(target_h.index.min(), target_h.index.max())
print(nbr_h.index.min(), nbr_h.index.max())


2013-12-31 23:00:00 2019-12-31 23:00:00
2014-01-01 00:00:00 2019-12-31 23:00:00


C:\Users\attafuro\AppData\Local\Temp\ipykernel_1152\169219894.py:1: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  target_h = target.resample("H").mean()
C:\Users\attafuro\AppData\Local\Temp\ipykernel_1152\169219894.py:2: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  nbr_h    = nbr.resample("H").mean()


In [36]:
df = target_h.add_suffix("_tgt").join(nbr_h.add_suffix("_nbr"), how="inner")

# Keep only relevant columns (you can add more later)
df = df[[f"{y_col}_tgt", f"{y_col}_nbr"]].copy()

df.head()


,WVHT_tgt,WVHT_nbr
datetime,,
2014-01-01 00:00:00,0.96,0.915
2014-01-01 01:00:00,0.89,0.840
2014-01-01 02:00:00,0.89,0.790
2014-01-01 03:00:00,0.92,0.735
2014-01-01 04:00:00,0.94,0.745


In [37]:
n = len(df)

train_size = int(0.7 * n)
val_size   = int(0.15 * n)

train_df = df.iloc[:train_size].copy()
val_df   = df.iloc[train_size:train_size + val_size].copy()
test_df  = df.iloc[train_size + val_size:].copy()

len(train_df), len(val_df), len(test_df)



(36808, 7887, 7889)

Missing observations at the target station were estimated using a regression-based spatial imputation approach, where relationships between neighboring and target stations were learned from historical data and applied only when the target observation was unavailable.

In [38]:
LAG = 1  # hours

for df_ in [train_df, val_df, test_df]:
    df_["WVHT_nbr_lag1"] = df_["WVHT_nbr"].shift(LAG)


In [39]:
train_df[["WVHT_tgt", "WVHT_nbr", "WVHT_nbr_lag1"]].head(5)


,WVHT_tgt,WVHT_nbr,WVHT_nbr_lag1
datetime,,,
2014-01-01 00:00:00,0.96,0.915,NaN
2014-01-01 01:00:00,0.89,0.840,0.915
2014-01-01 02:00:00,0.89,0.790,0.840
2014-01-01 03:00:00,0.92,0.735,0.790
2014-01-01 04:00:00,0.94,0.745,0.735


In [40]:
from sklearn.linear_model import LinearRegression

# Use only rows where both target and lagged neighbor exist
mask = (
    train_df["WVHT_tgt"].notna() &
    train_df["WVHT_nbr_lag1"].notna()
)

X_train = train_df.loc[mask, ["WVHT_nbr_lag1"]]
y_train = train_df.loc[mask, "WVHT_tgt"]

imp_model = LinearRegression()
imp_model.fit(X_train, y_train)

imp_model.coef_, imp_model.intercept_


(array([0.82657752]), 0.37098374848018567)

In [41]:
def impute_target(df, target_col, feature_col, model):
    df = df.copy()
    missing = df[target_col].isna() & df[feature_col].notna()
    df.loc[missing, target_col] = model.predict(df.loc[missing, [feature_col]])
    return df


In [42]:
train_imp = impute_target(train_df, "WVHT_tgt", "WVHT_nbr_lag1", imp_model)
val_imp   = impute_target(val_df,   "WVHT_tgt", "WVHT_nbr_lag1", imp_model)
test_imp  = impute_target(test_df,  "WVHT_tgt", "WVHT_nbr_lag1", imp_model)

In [43]:
train_df["WVHT_tgt"].isna().sum(), train_imp["WVHT_tgt"].isna().sum()


(4125, 1487)

Since we have that target is missing

neighbor is also missing

We handle this without leakage using a past rolling mean.

In [44]:
def fallback_past_rolling_mean(df, col, window=24):
    df = df.copy()
    roll = df[col].shift(1).rolling(window=window, min_periods=1).mean()
    df[col] = df[col].fillna(roll)
    return df


In [45]:
train_imp = fallback_past_rolling_mean(train_imp, "WVHT_tgt", 24)
val_imp   = fallback_past_rolling_mean(val_imp,   "WVHT_tgt", 24)
test_imp  = fallback_past_rolling_mean(test_imp,  "WVHT_tgt", 24)


In [46]:
train_imp["WVHT_tgt"].isna().sum(), val_imp["WVHT_tgt"].isna().sum(), test_imp["WVHT_tgt"].isna().sum()

(1381, 0, 1)

In [53]:
def missing_report(name, df_, col="WVHT_tgt"):
    n = len(df_)
    m = df_[col].isna().sum()
    print(f"{name:8s} | rows={n:6d} | missing={m:6d} | missing%={(m/n)*100:6.2f}%")

print("BEFORE imputation:")
missing_report("train", train_df)
missing_report("val",   val_df)
missing_report("test",  test_df)

print("\nAFTER imputation:")
missing_report("train", train_imp)
missing_report("val",   val_imp)
missing_report("test",  test_imp)


BEFORE imputation:
train    | rows= 36808 | missing=  4125 | missing%= 11.21%
val      | rows=  7887 | missing=  1351 | missing%= 17.13%
test     | rows=  7889 | missing=  2834 | missing%= 35.92%

AFTER imputation:
train    | rows= 36808 | missing=  1381 | missing%=  3.75%
val      | rows=  7887 | missing=     0 | missing%=  0.00%
test     | rows=  7889 | missing=     1 | missing%=  0.01%


Columns explained

start

The timestamp when the missing stretch begins

end

The timestamp when the missing stretch ends

duration_hours

The length of that missing stretch, measured in hours

In [54]:
def find_nan_gaps(df_, col="WVHT_tgt"):
    s = df_[col].isna().astype(int)
    change = s.diff().fillna(0)

    gap_starts = df_.index[change == 1]
    gap_ends   = df_.index[change == -1]

    # handle if series starts/ends with NaNs
    if s.iloc[0] == 1:
        gap_starts = gap_starts.insert(0, df_.index[0])
    if s.iloc[-1] == 1:
        gap_ends = gap_ends.append(pd.Index([df_.index[-1]]))

    gaps = pd.DataFrame({"start": gap_starts, "end": gap_ends})
    gaps["duration_hours"] = (gaps["end"] - gaps["start"]) / pd.Timedelta(hours=1)
    return gaps

gaps_train = find_nan_gaps(train_df, "WVHT_tgt")
gaps_val   = find_nan_gaps(val_df, "WVHT_tgt")
gaps_test  = find_nan_gaps(test_df, "WVHT_tgt")

print("Train gaps:", len(gaps_train))
print("Val gaps:", len(gaps_val))
print("Test gaps:", len(gaps_test))

gaps_train.head(10)


Train gaps: 480
Val gaps: 28
Test gaps: 24


,start,end,duration_hours
0,2014-01-02 01:00:00,2014-01-02 02:00:00,1.0
1,2014-01-02 14:00:00,2014-01-02 15:00:00,1.0
2,2014-01-03 03:00:00,2014-01-03 04:00:00,1.0
3,2014-01-03 08:00:00,2014-01-03 09:00:00,1.0
4,2014-01-05 20:00:00,2014-01-05 22:00:00,2.0
5,2014-01-06 01:00:00,2014-01-25 20:00:00,475.0
6,2014-01-26 07:00:00,2014-01-26 08:00:00,1.0
7,2014-01-27 00:00:00,2014-01-27 01:00:00,1.0
8,2014-01-27 04:00:00,2014-01-27 05:00:00,1.0
9,2014-01-27 18:00:00,2014-01-27 19:00:00,1.0


In [55]:
gaps_train.sort_values("duration_hours", ascending=False).head(15)

,start,end,duration_hours
230,2015-02-07 02:00:00,2015-04-06 15:00:00,1405.0
5,2014-01-06 01:00:00,2014-01-25 20:00:00,475.0
38,2014-02-19 18:00:00,2014-03-11 12:00:00,474.0
42,2014-03-13 03:00:00,2014-03-28 00:00:00,357.0
66,2014-04-11 18:00:00,2014-04-20 06:00:00,204.0
405,2017-04-06 12:00:00,2017-04-14 12:00:00,192.0
250,2015-07-22 03:00:00,2015-07-29 18:00:00,183.0
72,2014-04-24 13:00:00,2014-04-28 13:00:00,96.0
469,2017-11-07 11:00:00,2017-11-09 19:00:00,56.0
20,2014-02-11 09:00:00,2014-02-13 06:00:00,45.0


In [60]:
train_imp = train_imp.dropna(subset=["WVHT_tgt"])
val_imp   = val_imp.dropna(subset=["WVHT_tgt"])
test_imp  = test_imp.dropna(subset=["WVHT_tgt"])